# Build frozen dataset v3 on Kaggle

Supports a fresh parent-to-audit run and a resume flow: `semantic_audit_seed67_v3.zip -> freeze -> 6 configs -> package`. Resume does not download LaBSE and does not need a GPU.

In [ ]:
import os, subprocess, sys

# True when semantic_audit_seed67_v3.zip already exists.
RESUME_FROM_AUDIT_ZIP = True
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if not RESUME_FROM_AUDIT_ZIP:
    subprocess.run(['nvidia-smi'], check=False)
    if not torch.cuda.is_available():
        raise RuntimeError('Enable a GPU before running the LaBSE audit.')
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Resume from audit ZIP: freeze needs no GPU or LaBSE download.')

## 1. Tìm và giải nén project v3 parent


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, json, shutil, tempfile, zipfile

input_root = Path('/kaggle/input')
if not input_root.is_dir():
    raise FileNotFoundError('Missing /kaggle/input; use Add Input first.')

def is_project_root(path):
    return ((path / 'requirements-semantic.txt').is_file()
            and (path / 'src/data/finalize.py').is_file()
            and (path / 'artifacts/processed_token448_seed67_v3_parent/manifest.json').is_file())

def project_roots(root):
    values = [root] if is_project_root(root) else []
    values += [p.parent for p in root.rglob('requirements-semantic.txt') if is_project_root(p.parent)]
    return sorted(set(values), key=str)

def zip_contains_project(path):
    try:
        with zipfile.ZipFile(path) as archive:
            names = {name.replace('\\', '/').lstrip('/') for name in archive.namelist()}
        suffixes = ('requirements-semantic.txt', 'src/data/finalize.py',
                    'artifacts/processed_token448_seed67_v3_parent/manifest.json')
        return all(any(name.endswith(suffix) for name in names) for suffix in suffixes)
    except zipfile.BadZipFile:
        return False

def safe_extract(zip_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        corrupt = archive.testzip()
        if corrupt: raise ValueError(f'Corrupt ZIP entry {corrupt}: {zip_path}')
        for member in archive.infolist():
            relative = PurePosixPath(member.filename.replace('\\', '/'))
            if relative.is_absolute() or '..' in relative.parts or not relative.parts:
                raise ValueError(f'Unsafe ZIP path: {member.filename}')
            target = destination.joinpath(*relative.parts)
            if not target.resolve().is_relative_to(root):
                raise ValueError(f'Unsafe ZIP path: {member.filename}')
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, target.open('wb') as output:
                    shutil.copyfileobj(source, output)

work_root = Path(tempfile.mkdtemp(prefix='v3_builder_', dir='/kaggle/working'))
direct = project_roots(input_root)
project_zips = [p for p in input_root.rglob('*.zip') if zip_contains_project(p)]
if len(direct) == 1:
    PROJECT_DIR = work_root / 'project'
    shutil.copytree(direct[0], PROJECT_DIR)
elif not direct and len(project_zips) == 1:
    extract_dir = work_root / 'project'
    safe_extract(project_zips[0], extract_dir)
    roots = project_roots(extract_dir)
    if len(roots) != 1: raise ValueError(f'Expected one project root, found: {roots}')
    PROJECT_DIR = roots[0]
else:
    raise ValueError(f'Expected one project input. direct={direct}, project_zips={project_zips}')

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path: sys.path.insert(0, str(PROJECT_DIR))
print('Project:', PROJECT_DIR)

## 2. Cài dependency và model semantic

Chế độ online tự cài stack đã khóa và tải đúng LaBSE revision khi audit. Nếu bắt buộc offline, hãy upload một wheelhouse Linux và Hugging Face cache đã chứa `sentence-transformers/LaBSE@836121a`, rồi điền hai đường dẫn bên dưới.

In [ ]:
OFFLINE_MODE = False
WHEELHOUSE_DIR = ''
HF_CACHE_DIR = ''
INSTALL_SEMANTIC_STACK = not RESUME_FROM_AUDIT_ZIP
requirements = PROJECT_DIR / 'requirements-semantic.txt'

if INSTALL_SEMANTIC_STACK:
    if OFFLINE_MODE:
        wheelhouse = Path(WHEELHOUSE_DIR); cache = Path(HF_CACHE_DIR)
        if not wheelhouse.is_dir() or not cache.is_dir():
            raise FileNotFoundError('OFFLINE_MODE requires valid WHEELHOUSE_DIR and HF_CACHE_DIR')
        os.environ['HF_HOME'] = str(cache)
        os.environ['HF_HUB_OFFLINE'] = '1'
        os.environ['TRANSFORMERS_OFFLINE'] = '1'
        install = [sys.executable, '-m', 'pip', 'install', '-q', '--no-index',
                   '--find-links', str(wheelhouse), '-r', str(requirements)]
    else:
        install = [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)]
    subprocess.run(install, check=True)
    from importlib.metadata import version
    print('transformers:', version('transformers'))
    print('sentence-transformers:', version('sentence-transformers'))
    print('Semantic model: sentence-transformers/LaBSE @ 836121a')
else:
    print('Skipping semantic stack: the audit ZIP already contains all scores.')

## 3. Xác minh parent v3


In [ ]:
import pandas as pd
from IPython.display import display
PARENT_DIR = PROJECT_DIR / 'artifacts/processed_token448_seed67_v3_parent'
AUDIT_DIR = PROJECT_DIR / 'artifacts/processed_semantic_audit_seed67_v3'
FROZEN_DIR = PROJECT_DIR / 'artifacts/processed_clean_semantic_seed67_v3'
CONFIG_DIR = PROJECT_DIR / 'configs/generated_v3'
parent_manifest = json.loads((PARENT_DIR / 'manifest.json').read_text(encoding='utf-8'))
expected = {
    'dataset_version': 'token448_seed67_v3_parent',
    'maximum_chunk_tokens': 448,
    'tokenizer_model': 'vinai/vinai-translate-en2vi-v2',
    'tokenizer_revision': '1e47495b5dac16097b1d9e7b4133c27e1f40504f',
}
for key, value in expected.items():
    if parent_manifest.get('config', {}).get(key) != value:
        raise ValueError(f'Parent contract sai {key}: {parent_manifest.get("config", {}).get(key)}')
if parent_manifest.get('split_contract', {}).get('seed') != 67:
    raise ValueError('Parent v3 phải giữ split seed 67')
display(pd.DataFrame([parent_manifest['counts']]).T.rename(columns={0: 'records'}))
print('Parent contract: PASSED')


## 4. Resume from semantic audit ZIP

Attach `semantic_audit_seed67_v3.zip` as a Kaggle Input. This cell verifies checksums and restores the audit without running LaBSE again.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def validate_audit(directory):
    manifest_path = directory / 'manifest.json'
    report_path = directory / 'semantic_audit.json'
    if not manifest_path.is_file() or not report_path.is_file():
        raise FileNotFoundError(f'Audit is missing manifest/report: {directory}')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    finalization = manifest.get('finalization', {})
    if manifest.get('dataset_id') != 'legal_envi_semantic_audit_seed67_v3':
        raise ValueError(f'Sai audit dataset ID: {manifest.get("dataset_id")}')
    if manifest.get('semantic_audit_only') is not True:
        raise ValueError('Input must be a score-only semantic audit')
    if finalization.get('semantic_model') != 'sentence-transformers/LaBSE':
        raise ValueError('Sai semantic model')
    if finalization.get('semantic_model_revision') != '836121a':
        raise ValueError('Sai semantic model revision')
    if finalization.get('minimum_semantic_similarity') is not None:
        raise ValueError('Audit ZIP already has a threshold; expected score-only audit')
    for name, metadata in manifest.get('files', {}).items():
        file_path = directory / name
        if not file_path.is_file() or sha256_file(file_path) != metadata.get('sha256'):
            raise ValueError(f'Checksum audit sai: {name}')
    expected_parent_sha = sha256_file(PARENT_DIR / 'manifest.json')
    if manifest.get('parent_manifest_sha256') != expected_parent_sha:
        raise ValueError('Audit does not belong to this v3 parent')
    return manifest

if RESUME_FROM_AUDIT_ZIP:
    if (AUDIT_DIR / 'manifest.json').is_file():
        audit_manifest = validate_audit(AUDIT_DIR)
        print('Audit is already present; skipping extraction.')
    else:
        candidates = [p for p in input_root.rglob('*.zip')
                      if 'semanticauditseed67v3' in ''.join(ch for ch in p.name.casefold() if ch.isalnum())]
        if len(candidates) != 1:
            raise ValueError(f'Expected one semantic_audit_seed67_v3.zip, found: {candidates}')
        audit_extract = work_root / 'audit_extract'
        safe_extract(candidates[0], audit_extract)
        roots = []
        for manifest_path in audit_extract.rglob('manifest.json'):
            try:
                payload = json.loads(manifest_path.read_text(encoding='utf-8'))
            except (OSError, json.JSONDecodeError):
                continue
            if payload.get('dataset_id') == 'legal_envi_semantic_audit_seed67_v3':
                roots.append(manifest_path.parent)
        if len(roots) != 1:
            raise ValueError(f'Expected one audit root in ZIP, found: {roots}')
        shutil.copytree(roots[0], AUDIT_DIR)
        audit_manifest = validate_audit(AUDIT_DIR)
        print('Restored audit from:', candidates[0])
    print('AUDIT ZIP: PASSED | counts:', audit_manifest['counts'])
else:
    print('Fresh run selected: run the LaBSE audit cell below.')

## 4. Chạy LaBSE semantic audit

Bước này chỉ chấm score, chưa loại theo threshold. Có thể mất khá lâu; output hoàn chỉnh có `semantic_audit.json` và manifest mang cờ `semantic_audit_only=true`.

In [ ]:
RUN_SEMANTIC_AUDIT = False
SEMANTIC_BATCH_SIZE = 64
if RUN_SEMANTIC_AUDIT:
    if RESUME_FROM_AUDIT_ZIP:
        raise ValueError('Resume mode is active; do not rerun LaBSE')
    if AUDIT_DIR.exists() and any(AUDIT_DIR.iterdir()):
        raise FileExistsError(f'Audit output already exists: {AUDIT_DIR}')
    command = [sys.executable, '-u', '-m', 'src.data.finalize',
        '--input-dir', str(PARENT_DIR), '--output-dir', str(AUDIT_DIR),
        '--dataset-id', 'legal_envi_semantic_audit_seed67_v3',
        '--split-seed', '67', '--minimum-alignment-score', '0.75',
        '--semantic-model', 'sentence-transformers/LaBSE',
        '--semantic-model-revision', '836121a',
        '--semantic-batch-size', str(SEMANTIC_BATCH_SIZE), '--device', 'cuda', '--quiet']
    subprocess.run(command, cwd=PROJECT_DIR, check=True)
    audit_zip = shutil.make_archive('/kaggle/working/semantic_audit_seed67_v3', 'zip', root_dir=AUDIT_DIR)
    print('Audit ZIP:', audit_zip)

audit_report_path = AUDIT_DIR / 'semantic_audit.json'
if audit_report_path.is_file():
    audit_manifest = validate_audit(AUDIT_DIR)
    print('Audit ready:', audit_report_path)
else:
    print('Audit missing: run LaBSE or enable RESUME_FROM_AUDIT_ZIP.')

## 5. Duyệt report và mô phỏng threshold


In [ ]:
if not audit_report_path.is_file():
    raise FileNotFoundError('Hãy hoàn thành semantic audit trước')
audit_report = json.loads(audit_report_path.read_text(encoding='utf-8'))
summary_rows, bin_rows, lowest_rows = [], [], []
for split, split_data in audit_report['splits'].items():
    summary_rows.append({'split': split, 'records': split_data['records'],
        'truncated_rate': split_data['truncated_rate'],
        'minimum': split_data['minimum'], 'maximum': split_data['maximum']})
    for category, category_data in split_data['categories'].items():
        row = {'split': split, 'category': category, 'records': category_data['records'],
               'truncated_rate': category_data['truncated_rate']}
        row.update({key: category_data[key] for key in ('p05','p25','p50','p75','p95')})
        summary_rows.append(row)
        for score_bin, count in category_data['bin_counts'].items():
            bin_rows.append({'split': split, 'category': category, 'bin': score_bin, 'count': count})
        for example in category_data['lowest_examples']:
            lowest_rows.append({'split': split, 'category': category, **example})
display(pd.DataFrame(summary_rows))
display(pd.DataFrame(bin_rows))
display(pd.DataFrame(lowest_rows).sort_values(['score','split']).head(100))
CANDIDATE_THRESHOLDS = [0.60, 0.65, 0.70, 0.72, 0.75, 0.80]
threshold_rows = []
for split in ('train', 'dev'):
    path = AUDIT_DIR / f'segments_{split}_en2vi.jsonl'
    rows = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
    for threshold in CANDIDATE_THRESHOLDS:
        kept = sum(float(row['semantic_similarity']) >= threshold for row in rows)
        threshold_rows.append({'split': split, 'threshold': threshold, 'kept': kept,
            'removed': len(rows)-kept, 'kept_rate': 100*kept/len(rows)})
display(pd.DataFrame(threshold_rows))
REVIEW_THRESHOLD = 0.70  # chỉ để lấy mẫu duyệt, chưa phải threshold đã chốt
REVIEW_WINDOW = 0.015
REVIEW_MAX_ROWS = 100
near_threshold = []
for split in ('train', 'dev'):
    path = AUDIT_DIR / f'segments_{split}_en2vi.jsonl'
    for line in path.read_text(encoding='utf-8').splitlines():
        if not line.strip(): continue
        row = json.loads(line); score = float(row['semantic_similarity'])
        if abs(score - REVIEW_THRESHOLD) <= REVIEW_WINDOW:
            near_threshold.append({'split': split, 'category': row.get('category'),
                'segment_id': row.get('segment_id'), 'score': score,
                'source': row.get('source'), 'target': row.get('target')})
near_frame = pd.DataFrame(near_threshold).sort_values(['score','split']) if near_threshold else pd.DataFrame()
display(near_frame.head(REVIEW_MAX_ROWS))
print('Mẫu quanh ngưỡng:', len(near_threshold), '| đang hiển thị tối đa', REVIEW_MAX_ROWS)
print('Use train/dev only for threshold selection; exclude public test.')
print('Cosine score không phải xác suất. Đọc source/target quanh ngưỡng trước khi freeze.')

## 6. Freeze v3 và sinh sáu config

Điền threshold sau khi duyệt. Generation parameters bên dưới phải phản ánh decoding đã chốt trên dev; có thể sửa config sau nhưng không được dùng public test để chọn chúng. Freeze tái sử dụng score, không chạy LaBSE lần hai.

In [ ]:
RUN_FREEZE = False
SEMANTIC_THRESHOLD = 0.70
LORA_LEARNING_RATE = 5e-5
CHECKPOINT_SAMPLES = 256
GENERATION_LENGTH_PENALTY = 1.0
GENERATION_REPETITION_PENALTY = 1.0
GENERATION_MAX_NEW_TOKENS = 448
required_configs = {
    'en2vi_zero_shot.json','en2vi_lora.json','en2vi_full.json',
    'vi2en_zero_shot.json','vi2en_lora.json','vi2en_full.json'}

if RUN_FREEZE:
    threshold = float(SEMANTIC_THRESHOLD)
    if not -1.0 <= threshold <= 1.0:
        raise ValueError('Threshold must be in [-1,1]')
    frozen_manifest_path = FROZEN_DIR / 'manifest.json'
    if frozen_manifest_path.is_file():
        frozen_manifest = json.loads(frozen_manifest_path.read_text(encoding='utf-8'))
        recorded = frozen_manifest.get('finalization', {}).get('minimum_semantic_similarity')
        if (frozen_manifest.get('dataset_id') != 'legal_envi_clean_semantic_seed67_v3'
                or frozen_manifest.get('semantic_audit_only') is not False
                or recorded != threshold):
            raise ValueError(f'Existing frozen output does not match threshold {threshold}: {recorded}')
        print('Frozen dataset is valid; skipping freeze.')
    else:
        if FROZEN_DIR.exists() and any(FROZEN_DIR.iterdir()):
            raise FileExistsError(f'Incomplete frozen output requires inspection: {FROZEN_DIR}')
        freeze = [sys.executable, '-u', '-m', 'src.data.finalize',
            '--input-dir', str(AUDIT_DIR), '--output-dir', str(FROZEN_DIR),
            '--dataset-id', 'legal_envi_clean_semantic_seed67_v3',
            '--split-seed', '67', '--minimum-alignment-score', '0.75',
            '--semantic-model', 'sentence-transformers/LaBSE',
            '--semantic-model-revision', '836121a',
            '--minimum-semantic-similarity', str(threshold),
            '--reuse-semantic-scores', '--quiet']
        subprocess.run(freeze, cwd=PROJECT_DIR, check=True)

    existing_configs = {p.name for p in CONFIG_DIR.glob('*.json')}
    if not required_configs.issubset(existing_configs):
        create = [sys.executable, '-u', '-m', 'src.vinai_baseline.create_configs',
            '--project-root', str(PROJECT_DIR), '--dataset-dir', str(FROZEN_DIR),
            '--output-dir', str(CONFIG_DIR), '--learning-rate', str(LORA_LEARNING_RATE),
            '--checkpoint-samples', str(CHECKPOINT_SAMPLES),
            '--length-penalty', str(GENERATION_LENGTH_PENALTY),
            '--repetition-penalty', str(GENERATION_REPETITION_PENALTY),
            '--max-new-tokens', str(GENERATION_MAX_NEW_TOKENS), '--overwrite']
        subprocess.run(create, cwd=PROJECT_DIR, check=True)

if (FROZEN_DIR / 'manifest.json').is_file():
    frozen_manifest = json.loads((FROZEN_DIR / 'manifest.json').read_text(encoding='utf-8'))
    existing_configs = {p.name for p in CONFIG_DIR.glob('*.json')}
    if frozen_manifest.get('semantic_audit_only'):
        raise ValueError('Frozen manifest cannot be audit-only')
    print('Frozen dataset:', frozen_manifest['dataset_id'])
    print('Threshold:', frozen_manifest['finalization']['minimum_semantic_similarity'])
    display(pd.DataFrame([frozen_manifest['counts']]).T.rename(columns={0:'records'}))
    print('Missing configs:', sorted(required_configs - existing_configs))
    if RUN_FREEZE and not required_configs.issubset(existing_configs):
        raise FileNotFoundError('Six v3 configs were not created')
    if RUN_FREEZE:
        print('FREEZE + SIX CONFIGS: PASSED')
else:
    print('Set RUN_FREEZE=True to freeze and create configs.')

## 7. Kiểm tra và đóng gói frozen v3 để train


In [ ]:
RUN_PACKAGE = False  # bật sau khi freeze thành công
if RUN_PACKAGE:
    manifest_path = FROZEN_DIR / 'manifest.json'
    if not manifest_path.is_file(): raise FileNotFoundError('Thiếu frozen manifest')
    frozen_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if frozen_manifest.get('dataset_id') != 'legal_envi_clean_semantic_seed67_v3':
        raise ValueError('Sai frozen dataset ID')
    required_configs = {'en2vi_zero_shot.json','en2vi_lora.json','en2vi_full.json',
        'vi2en_zero_shot.json','vi2en_lora.json','vi2en_full.json'}
    if not required_configs.issubset({p.name for p in CONFIG_DIR.glob('*.json')}):
        raise FileNotFoundError('Chưa sinh đủ sáu config v3')
    test_targets = [
        'tests.test_data_preparation',
        'tests.test_offline_modeling',
        'tests.test_vinai_baseline.TokenAuditTests',
        'tests.test_vinai_baseline.LegalMetricTests',
        'tests.test_vinai_baseline.ComparisonTests',
        'tests.test_vinai_baseline.OutputSafetyTests',
        'tests.test_vinai_baseline.DocumentInferenceTests',
        'tests.test_vinai_baseline.KaggleNotebookTests',
        'tests.test_submission']
    subprocess.run([sys.executable, '-m', 'unittest', *test_targets, '-q'],
                   cwd=PROJECT_DIR, check=True)
    frozen_only_zip = shutil.make_archive('/kaggle/working/processed_clean_semantic_seed67_v3',
        'zip', root_dir=FROZEN_DIR)
    bundle_path = Path('/kaggle/working/NLP_v3_frozen.zip')
    include = ['README','requirements.txt','requirements-training.txt',
        'requirements-training-colab.txt','requirements-semantic.txt','configs','src','tests',
        'docs','scripts','submission','data/instruction.md',
        'notebooks/03_vinai_kaggle.ipynb','notebooks/04_v3_builder_kaggle.ipynb',
        'artifacts/processed_clean_semantic_seed67_v3']
    with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
        for relative_text in include:
            source = PROJECT_DIR / relative_text
            if not source.exists(): raise FileNotFoundError(source)
            paths = [source] if source.is_file() else [p for p in source.rglob('*') if p.is_file()]
            for path in paths:
                if '__pycache__' in path.parts or path.suffix in {'.pyc','.pyo'}: continue
                archive.write(path, path.relative_to(PROJECT_DIR).as_posix())
    def sha256(path):
        digest = hashlib.sha256()
        with Path(path).open('rb') as handle:
            for block in iter(lambda: handle.read(1024*1024), b''): digest.update(block)
        return digest.hexdigest()
    checksums = {Path(frozen_only_zip).name: sha256(frozen_only_zip), bundle_path.name: sha256(bundle_path)}
    checksum_path = Path('/kaggle/working/v3_sha256.json')
    checksum_path.write_text(json.dumps(checksums, indent=2), encoding='utf-8')
    print('Frozen-only ZIP:', frozen_only_zip)
    print('Training bundle:', bundle_path, f'{bundle_path.stat().st_size/1024/1024:.2f} MB')
    print('Checksums:', checksum_path, checksums)
else:
    print('Sau khi freeze: đặt RUN_PACKAGE=True để kiểm tra và tạo ZIP.')


## Outputs to save

Download `NLP_v3_frozen.zip`, `processed_clean_semantic_seed67_v3.zip`, and `v3_sha256.json`. For training, use `NLP_v3_frozen.zip`, set `DATASET_PROFILE="v3_frozen"`, run smoke before full training, and evaluate zero-shot before LoRA. Never use public test to select a model or decoding settings.